# Streaming Agen: Solusi Latihan

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dteti-workshop/workshop-agent-migas/blob/main/colab/solusi/hari3/bonus-streaming-agen-solusi.ipynb)

Ini **solusi lengkap** semua latihan bab ini (2 latihan) di Google Colab. Semua bagian `TODO` sudah terisi, jadi notebook ini tinggal dijalankan (**Runtime > Run all**, atau `Shift+Enter` per cell). Keluaran dari satu kali eksekusi dan penjelasan tiap latihan sudah disertakan; jawaban model berubah tiap kali dijalankan, jadi hasil Anda bisa sedikit berbeda. Kerjakan dulu [notebook latihannya](https://colab.research.google.com/github/dteti-workshop/workshop-agent-migas/blob/main/colab/hari3/bonus-streaming-agen-latihan.ipynb); buka solusi ini untuk mencocokkan hasil Anda atau saat macet.

1. Pilih **File > Save a copy in Drive** agar perubahan tersimpan di akun Anda sendiri.
2. Jalankan tiga cell **Persiapan Colab** berurutan (sekitar 2-3 menit pada sesi baru).
3. Kerjakan dari atas ke bawah dengan `Shift+Enter`.

Kalau runtime terputus, file di `/content` (termasuk `.env`) hilang. Jalankan ulang Persiapan, lalu lanjutkan dari cell yang sedang Anda kerjakan.

## Persiapan Colab
Jalankan tiga cell berikut berurutan sebelum cell lain di notebook ini.

In [ ]:
# Persiapan 1 - Menyalin repository dan memasang pustaka (sekitar 2-3 menit pada sesi baru)
import os
import subprocess
import sys

REPO_URL = "https://github.com/dteti-workshop/workshop-agent-migas.git"
REPO = "/content/workshop-agent-migas" if os.path.isdir("/content") else os.path.abspath("workshop-agent-migas")
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", "-b", "main", REPO_URL, REPO], check=True)
os.chdir(REPO)

pasang = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "colab/requirements-colab.txt", "-e", "."],
                        capture_output=True, text=True)
if pasang.returncode != 0:
    print(pasang.stderr[-3000:])
    raise RuntimeError("Instalasi pustaka gagal. Jalankan ulang cell ini.")

# "-e ." memasang common dan migas_shared agar subproses (server MCP) dapat mengimpornya; kernel yang sedang berjalan
# belum memuat instalasi baru, jadi folder repo juga ditambahkan ke sys.path
if REPO not in sys.path:
    sys.path.insert(0, REPO)
os.environ["PYTHONPATH"] = REPO + os.pathsep + os.environ.get("PYTHONPATH", "")
print("Repository dan pustaka siap. Folder kerja:", os.getcwd())

In [ ]:
# Persiapan 2 - Mengisi ID peserta dan API key (dilewati bila .env sudah berisi key)
import re
from getpass import getpass
from pathlib import Path

from dotenv import load_dotenv

env = Path(REPO) / ".env"


def rahasia(nama: str):
    """Nilai dari Colab Secrets (ikon kunci di panel kiri), atau None bila tidak ada."""
    try:
        from google.colab import userdata
        return userdata.get(nama)
    except Exception:  # noqa: BLE001 - bukan di Colab, secret belum dibuat, atau akses notebook belum diizinkan
        return None


if not (env.exists() and re.search(r"(?m)^ONLINE_API_KEY=(?!isi-dengan)\S+", env.read_text(encoding="utf-8"))):
    peserta = (rahasia("PARTICIPANT_ID") or input("ID peserta (nomor meja dan nama singkat, contoh P07-andi): ")).strip()
    kunci = (rahasia("ONLINE_API_KEY") or getpass("API key workshop (ketikan tidak ditampilkan): ")).strip()
    if not kunci:
        raise ValueError("API key belum diisi. Jalankan ulang cell ini.")
    isi = (Path(REPO) / ".env.example").read_text(encoding="utf-8")
    for nama, nilai in (("PARTICIPANT_ID", peserta or "P00-nama"), ("ONLINE_API_KEY", kunci)):
        isi = re.sub(rf"(?m)^{nama}=.*$", lambda _, n=nama, v=nilai: f"{n}={v}", isi)
    env.write_text(isi, encoding="utf-8")
    del kunci

# Nilai dari .env dimuat ulang; modul yang sudah diimpor dibuang agar membaca nilai terbaru
load_dotenv(env, override=True)
for modul in [m for m in sys.modules if m.split(".")[0] in ("common", "migas_shared", "rag_pipeline")]:
    del sys.modules[modul]
print("File .env siap untuk peserta", os.environ.get("PARTICIPANT_ID"))

In [ ]:
# Persiapan 3 - Memeriksa koneksi ke model
from migas_shared.llm import chat

try:
    print("Jawaban model:", chat("Balas hanya dengan kata: SIAP").strip())
except Exception as error:  # noqa: BLE001
    raise RuntimeError("Model tidak menjawab. Periksa API key (jalankan ulang Persiapan 2) dan koneksi internet. "
                       f"Pesan asli: {error}") from error

## Bab bonus Streaming Agen - Solusi Latihan 1: Asisten Operasi Langsung
Jalankan sel satu per satu dengan Shift+Enter

Asisten terminal dengan streaming yang menggabungkan tiga ide:
- Teks langsung: jawaban muncul kata demi kata (`stream_mode="messages"`)
- Status langsung: baris status ketika tool mulai atau melapor progres (`stream_mode="updates"` dan `"custom"`)
- Memori: follow-up question berfungsi (checkpointer dari bab 5)

Bagian utama memakai empat pertanyaan berskrip, satu sel per giliran. Sel terakhir membuka mode
interaktif (ketik pertanyaan sendiri) setelah `INTERAKTIF` diubah menjadi `True`.

In [ ]:
# Langkah 1 - Menyiapkan model
import os
import time

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.messages import AIMessage, AIMessageChunk, HumanMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.config import get_stream_writer

from migas_shared.field_tools import DATA_SUMUR, cek_status_sumur

load_dotenv()

model = ChatOpenAI(
    model=os.getenv("ONLINE_MODEL"),
    base_url=os.getenv("ONLINE_BASE_URL"),
    api_key=os.getenv("ONLINE_API_KEY"),
    reasoning_effort=os.getenv("LLM_REASONING_EFFORT", "none"),
)

## Langkah 2 - Tool audit dan agen dengan memori
`audit_water_cut` melapor progres lewat `get_stream_writer()`. `InMemorySaver` menjaga riwayat per `thread_id`.

In [ ]:
NAMA_RAMAH = {
    "cek_status_sumur": "Memeriksa status sumur",
    "audit_water_cut": "Mengaudit water cut semua sumur",
}


@tool
def audit_water_cut() -> str:
    """Audit water cut SEMUA sumur satu per satu dan sebutkan yang melewati 75%. Lambat: pakai hanya untuk audit penuh."""
    writer = get_stream_writer()
    tinggi = []
    for nomor, (kode, data) in enumerate(DATA_SUMUR.items(), start=1):
        writer({"progres": f"memeriksa {nomor}/{len(DATA_SUMUR)} sumur ({kode})"})
        time.sleep(0.7)
        total = data["minyak_bopd"] + data["air_bwpd"]
        if total and data["air_bwpd"] / total * 100 > 75:
            tinggi.append(f"{kode} ({data['air_bwpd'] / total * 100:.0f}%)")
    return f"Diaudit {len(DATA_SUMUR)} sumur. Water cut di atas 75%: {', '.join(tinggi) or 'tidak ada'}"


agent = create_agent(
    model,
    tools=[cek_status_sumur, audit_water_cut],
    system_prompt="Anda asisten operasi. Pakai tool bila perlu. Jawab singkat dalam Bahasa Indonesia.",
    checkpointer=InMemorySaver(),
)
config = {"configurable": {"thread_id": "sesi-langsung"}}

## Langkah 3 - Fungsi untuk satu giliran
`tanya_langsung` mengalirkan satu jawaban ke terminal dengan status langsung dan teks langsung.

In [ ]:
def tanya_langsung(agent, teks: str, config: dict) -> None:
    """Alirkan satu jawaban ke terminal dengan status langsung dan teks langsung."""
    dalam_jawaban = False
    for mode, data in agent.stream(
        {"messages": [HumanMessage(content=teks)]}, config, stream_mode=["updates", "messages", "custom"]
    ):
        if mode == "updates":
            for node, isi in data.items():
                for pesan in isi.get("messages", []) if isinstance(isi, dict) else []:
                    if isinstance(pesan, AIMessage) and pesan.tool_calls:
                        for call in pesan.tool_calls:
                            print(f"   {NAMA_RAMAH.get(call['name'], call['name'])}...")
                    elif node == "tools":
                        print(f"   {pesan.name} selesai")
        elif mode == "custom":
            print(f"      -> {data.get('progres', data)}")
        elif mode == "messages":
            chunk, metadata = data
            if isinstance(chunk, AIMessageChunk) and metadata["langgraph_node"] == "model" and chunk.text:
                if not dalam_jawaban:
                    print("Agen: ", end="", flush=True)
                    dalam_jawaban = True
                print(chunk.text, end="", flush=True)
    print("\n")

## Langkah 4 - Giliran 1: status sumur pertama

In [ ]:
teks_1 = "Bagaimana status sumur SR-020?"
print(f"Pengguna: {teks_1}")
tanya_langsung(agent, teks_1, config)

Pengguna: Bagaimana status sumur SR-020?
Agen: 

   Memeriksa status sumur...
   cek_status_sumur selesai
Berikut status sumur **SR-020**:

- **Area:** Seruni
- **Jenis Pompa:** ESP (Electric Submersible Pump)
- **Status:** Produksi
- **Produksi Minyak:** 120 bopd
- **Produksi Air:** 480 bwpd
- **Catatan:** ESP sempat *trip overload* kemarin sore, namun sudah stabil sejak pukul 18.00.

Secara umum sumur beroperasi normal, namun perlu dipantau karena ada catatan trip overload baru-baru ini.



## Langkah 5 - Giliran 2: status sumur kedua

In [ ]:
teks_2 = "Dan sumur ML-012?"
print(f"Pengguna: {teks_2}")
tanya_langsung(agent, teks_2, config)

Pengguna: Dan sumur ML-012?
Agen: 

   Memeriksa status sumur...
   cek_status_sumur selesai
Berikut status sumur **ML-012**:

- **Area:** Melati
- **Jenis Pompa:** ESP
- **Status:** Produksi
- **Produksi Minyak:** 85 bopd
- **Produksi Air:** 340 bwpd
- **Catatan:** Normal

Sumur dalam kondisi normal dan tidak ada masalah.



## Langkah 6 - Giliran 3: audit water cut semua sumur
Tool ini lambat dan melapor progres lewat aliran `custom`.

In [ ]:
teks_3 = "Jalankan audit water cut semua sumur dan beri tahu yang tinggi."
print(f"Pengguna: {teks_3}")
tanya_langsung(agent, teks_3, config)

Pengguna: Jalankan audit water cut semua sumur dan beri tahu yang tinggi.
Agen: 

   Mengaudit water cut semua sumur...
      -> memeriksa 1/4 sumur (SR-020)
      -> memeriksa 2/4 sumur (SR-027)
      -> memeriksa 3/4 sumur (KN-033)
      -> memeriksa 4/4 sumur (ML-012)
   audit_water_cut selesai
Hasil audit water cut seluruh sumur:

- **Total sumur diaudit:** 4
- **Sumur yang melewati batas 75%:**

| Sumur | Water Cut |
|-------|-----------|
| **SR-020** | 80% |
| **SR-027** | 80% |
| **ML-012** | 80% |

Ketiga sumur tersebut memiliki water cut **80%**, melampaui ambang batas 75%. Perlu perhatian untuk penanganan produksi air yang berlebih pada sumur-sumur tersebut.



## Langkah 7 - Giliran 4: pertanyaan lanjutan
Berkat checkpointer, agen mengingat dua sumur yang ditanyakan di awal.

In [ ]:
teks_4 = "Dari dua sumur yang pertama saya tanyakan, mana yang produksi minyaknya lebih besar?"
print(f"Pengguna: {teks_4}")
tanya_langsung(agent, teks_4, config)

Pengguna: Dari dua sumur yang pertama saya tanyakan, mana yang produksi minyaknya lebih besar?
Agen: Dari dua sumur yang pertama Anda tanyakan:

- **SR-020:** 120 bopd
- **ML-012:** 85 bopd

Jadi, **SR-020** memiliki produksi minyak lebih besar (120 bopd vs 85 bopd).



## Langkah 8 - Mencoba sendiri (opsional)
Ubah `INTERAKTIF` menjadi `True`, jalankan sel, lalu ketik pertanyaan. Ketik `keluar` (atau kosongkan) untuk berhenti.
Percakapan melanjutkan thread yang sama.

In [ ]:
INTERAKTIF = False  # ubah ke True untuk mencoba sendiri

if INTERAKTIF:
    print("Tanyakan status sumur atau minta audit water cut. Ketik 'keluar' untuk berhenti.\n")
    while True:
        teks = input("Pengguna: ").strip()
        if teks.lower() in {"keluar", "exit", ""}:
            break
        tanya_langsung(agent, teks, config)

### Penjelasan Solusi Bonus Streaming 1: asisten operasi langsung

`agent.stream` dengan tiga mode sekaligus: `updates` untuk kejadian per langkah agen, `custom` untuk pesan yang dikirim tool sendiri, dan `messages` untuk token jawaban.

- `TODO(BS.1)`: `audit_water_cut` memanggil `get_stream_writer()` dan mengirim progres per sumur sebelum `time.sleep(0.7)`. Jeda itu mensimulasikan tool yang lambat.
- `TODO(BS.2)`: `checkpointer=InMemorySaver()` dan `thread_id` yang sama membuat pertanyaan lanjutan bisa dijawab.
- `TODO(BS.3)`: `stream_mode=["updates", "messages", "custom"]` menghasilkan pasangan `(mode, data)`. Teks hanya dicetak untuk chunk dari node `model`, supaya keluaran tool tidak ikut tercetak sebagai jawaban.

Pada giliran ketiga terlihat baris `memeriksa 1/4 sumur ... 4/4` muncul satu per satu sebelum jawaban akhir; pengguna tidak menatap layar kosong selama audit berjalan. Audit menemukan SR-020, SR-027, dan ML-012 di atas 75% (masing-masing 80%); KN-033 tidak terhitung karena tidak berproduksi. Giliran keempat ("dari dua sumur yang pertama saya tanyakan") dijawab benar tanpa tool karena riwayat tersimpan.

## Solusi Latihan 2 (Bonus): Endpoint HTTP Streaming (SSE)

Latihan ini berupa file Python, `hari3/bonus-streaming-agen/latihan_2.py`. Di Colab file itu ditulis dari cell di bawah. Kode di bawah sudah lengkap. Jalankan cell untuk menyimpan file.

Baris pertama cell (`%%writefile ...`) menentukan nama file dan tidak boleh diubah.

In [ ]:
%%writefile hari3/bonus-streaming-agen/latihan_2.py
"""
Bab bonus Streaming Agen - Latihan Bonus: Endpoint HTTP Streaming (SSE)

Server web kecil, hanya dengan library standar Python, yang mengalirkan jawaban agen ke klien sebagai
Server-Sent Events (SSE):

    GET /chat?q=<pertanyaan>   ->   data: {"type": "token", "text": "..."}\\n\\n   (berulang)
                                    data: {"type": "done"}\\n\\n

Browser dapat membacanya dengan API EventSource bawaan; di sini klien Python yang melakukannya.
(Di proyek nyata Anda memakai framework seperti FastAPI, tetapi gagasannya identik.)

Jalankan demo mandiri:  python hari3/bonus-streaming-agen/latihan_2.py --demo
Jalankan server saja:   python hari3/bonus-streaming-agen/latihan_2.py
                        lalu buka http://localhost:8000/chat?q=Apa%20itu%20water%20cut
"""

import json
import os
import sys
import threading
import urllib.parse
import urllib.request
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.messages import AIMessageChunk, HumanMessage
from langchain_openai import ChatOpenAI

load_dotenv()


def buat_agen():
    model = ChatOpenAI(
        model=os.getenv("ONLINE_MODEL"),
        base_url=os.getenv("ONLINE_BASE_URL"),
        api_key=os.getenv("ONLINE_API_KEY"),
        reasoning_effort=os.getenv("LLM_REASONING_EFFORT", "none"),
    )
    return create_agent(
        model, tools=[], system_prompt="Anda asisten operasi produksi migas. Jawab dalam 2-3 kalimat pendek Bahasa Indonesia."
    )


AGEN = buat_agen()


def event_sse(payload: dict) -> bytes:
    """Format satu Server-Sent Event."""
    return f"data: {json.dumps(payload)}\n\n".encode("utf-8")


class PenanganChat(BaseHTTPRequestHandler):
    def do_GET(self):
        url = urllib.parse.urlparse(self.path)
        if url.path != "/chat":
            self.send_error(404, "Pakai /chat?q=pertanyaan+anda")
            return
        pertanyaan = urllib.parse.parse_qs(url.query).get("q", [""])[0].strip()
        if not pertanyaan:
            self.send_error(400, "Parameter ?q= belum diisi")
            return

        self.send_response(200)
        self.send_header("Content-Type", "text/event-stream")  # memberi tahu klien bahwa ini aliran SSE
        self.send_header("Cache-Control", "no-cache")
        self.end_headers()

        try:
            for chunk, metadata in AGEN.stream(
                {"messages": [HumanMessage(content=pertanyaan)]}, stream_mode="messages"
            ):
                if isinstance(chunk, AIMessageChunk) and metadata["langgraph_node"] == "model" and chunk.text:
                    self.wfile.write(event_sse({"type": "token", "text": chunk.text}))
                    self.wfile.flush()  # dorong setiap event keluar segera
            self.wfile.write(event_sse({"type": "done"}))
        except (BrokenPipeError, ConnectionResetError):
            pass  # klien menutup koneksi; hentikan streaming

    def log_message(self, format, *args):
        pass  # jaga konsol tetap tenang


def jalankan_klien(port: int, pertanyaan: str) -> None:
    """Klien SSE minimal: baca respons baris demi baris saat tiba."""
    url = f"http://127.0.0.1:{port}/chat?q={urllib.parse.quote(pertanyaan)}"
    print(f"Permintaan klien: GET {url}\n")
    print("Agen: ", end="", flush=True)
    jumlah = 0
    with urllib.request.urlopen(url) as response:
        for baris_mentah in response:
            baris = baris_mentah.decode("utf-8").strip()
            if not baris.startswith("data:"):
                continue  # baris pemisah kosong
            event = json.loads(baris[len("data:"):])
            jumlah += 1
            if event["type"] == "token":
                print(event["text"], end="", flush=True)
            elif event["type"] == "done":
                print(f"\n\nAliran selesai setelah {jumlah} event")


def main():
    if "--demo" in sys.argv:
        server = ThreadingHTTPServer(("127.0.0.1", 0), PenanganChat)  # port 0 = pilih port kosong
        port = server.server_address[1]
        threading.Thread(target=server.serve_forever, daemon=True).start()
        print(f"Server berjalan di http://127.0.0.1:{port}\n")
        jalankan_klien(port, "Apa bedanya alarm H2S tingkat 1 dan tingkat 2?")
        server.shutdown()
        return

    server = ThreadingHTTPServer(("127.0.0.1", 8000), PenanganChat)
    print("Mendengarkan di http://localhost:8000/chat?q=... (Ctrl+C untuk berhenti)")
    try:
        server.serve_forever()
    except KeyboardInterrupt:
        print("\nDihentikan")


if __name__ == "__main__":
    main()

Overwriting hari3/bonus-streaming-agen/latihan_2.py


Setelah file tersimpan, jalankan demo mandiri:

In [ ]:
!{sys.executable} hari3/bonus-streaming-agen/latihan_2.py --demo

Server berjalan di http://127.0.0.1:57298

Permintaan klien: GET http://127.0.0.1:57298/chat?q=Apa%20bedanya%20alarm%20H2S%20tingkat%201%20dan%20tingkat%202%3F

Agen: Alarm H2S tingkat 1 menandakan konsentrasi gas rendah (biasanya 10-20 ppm) dengan tindakan berupa peningkatan kewaspadaan dan pengecekan area. Alarm tingkat 2 menunjukkan kadar berbahaya (biasanya di atas 20 ppm) yang mewajibkan evakuasi segera dan penggunaan alat pelindung pernapasan.

Aliran selesai setelah 35 event


### Penjelasan Solusi Bonus Streaming 2: endpoint SSE

Server web kecil dari pustaka standar Python yang mengalirkan jawaban agen sebagai Server-Sent Events. Browser membacanya dengan `EventSource`; di sini klien Python melakukannya.

- `TODO(BS.4)`: satu event berformat `data: <JSON>` diakhiri dua baris baru, dikirim sebagai bytes UTF-8. Dua baris baru itu pemisah antar event.
- `TODO(BS.5)`: `AGEN.stream(..., stream_mode="messages")` dibaca dalam loop. Tiap potongan teks dari node `model` ditulis sebagai event `token` dan langsung `flush()`, kalau tidak, server menahan data di buffer dan klien menerima semuanya sekaligus. Event `done` menutup aliran.

Mode `--demo` memilih port kosong (`port 0`) agar tidak bentrok, menjalankan server di thread latar, lalu klien meminta satu pertanyaan. Pada keluaran, aliran selesai setelah puluhan event token. Ada satu catatan isi: agen ini tidak punya tool dan tidak membaca SOP, jadi angka ppm pada jawabannya berasal dari pengetahuan umum model dan bisa berbeda dari SOP-HSE-001. Latihan ini tentang mekanisme alirannya, bukan tentang kebenaran jawaban.